In [1]:
import torch
from components.datasets import load_dataset
from components.models import init_mf_params, create_anfis_model
from components.trainer import train_model


# Config
config = {
    "dataset": "moons", # options: mackey-glass, circles, spiral, moons, swiss-roll, iris, digits
    "mf_type": "advanced", # options: simple, advanced
    "mf_init": "random_uniform", # options: fcm, random_uniform
    "K": 2,
    "s_mode": "alpha_beta",
    "n_epochs": 1000,
    "lr": 0.001,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,
}

# 1) Load dataset
X_train, X_test, y_train, y_test, task_type, n_outputs = load_dataset(config["dataset"])

# 2) Initialize MF
centers_init, spreads_init = init_mf_params(
    X_train, K=config["K"], method=config["mf_init"], scale=1.0, seed=config["seed"]
)

# 3) Create model
model = create_anfis_model(
    config["mf_type"], centers_init, spreads_init, n_outputs=n_outputs, s_mode=config["s_mode"]
).to(config["device"])



/home/nazanin/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:174: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
from components.utils import evaluate_anfis_model

# 4) Train
model = train_model(
    model,
    X_train, y_train,
    X_test, y_test,
    task_type=task_type,
    n_outputs=n_outputs,
    n_epochs=config["n_epochs"],
    lr=config["lr"],
    batch_size=config.get("batch_size", 128),
    patience=config.get("patience", 30),
    device=config["device"],
)


# 5) Evaluate and plot results
evaluate_anfis_model(
    model, X_train, y_train, X_test, y_test, 
    device=config["device"], 
    s_mode_choice=config["s_mode"], 
    centers_init=centers_init,
    dataset_name=config["dataset"] 
)

epoch    1 | train_loss=0.714316 | val_loss=0.709790, acc=0.5000
epoch  100 | train_loss=0.328326 | val_loss=0.332110, acc=0.8450
epoch  200 | train_loss=0.269040 | val_loss=0.275207, acc=0.8800
epoch  300 | train_loss=0.260639 | val_loss=0.266616, acc=0.8900
epoch  400 | train_loss=0.255516 | val_loss=0.260106, acc=0.8950
epoch  500 | train_loss=0.244618 | val_loss=0.243251, acc=0.8900
epoch  600 | train_loss=0.089868 | val_loss=0.090189, acc=0.9800
epoch  700 | train_loss=0.049058 | val_loss=0.057794, acc=0.9800
epoch  800 | train_loss=0.033580 | val_loss=0.042994, acc=0.9850
epoch  900 | train_loss=0.024989 | val_loss=0.034524, acc=0.9900
epoch 1000 | train_loss=0.019306 | val_loss=0.028484, acc=0.9950

Final Results:
Mode: alpha_beta
Train: accuracy rate 0.999 (799/800) | error 0.001 (1)
Test: accuracy rate 0.995 (199/200) | error 0.005 (1)

Per-rule, per-feature MF contributions and raw participation:
Rule 0, Feature 0: Raw weight -> Gaussian=0.9985, Sigmoid=0.0015
Rule 0, Feature

IndexError: index 2 is out of bounds for axis 1 with size 2